### **Problem Statement**
A retail chain with 60 stores across 20 Kenyan counties has transaction data but no unified view of what drives its sales. It doesn’t know which stores, categories and customers generate the most revenue, whether its promotions create real incremental sales or just discount purchases that would have happened anyway, or how to plan demand.

#### **Objectives**
- Which counties, store formats, and product categories are our primary drivers of revenue and sales volume?
- What are the long-term sales trends over time, and how do sales fluctuate by the day of the week?
- What is the exact lift in units and revenue generated by our promotions?
- How does promotional performance vary across different discount depths and product categories?
- Which specific promotions are failing to generate a positive return and are not worth running in the future?
- How can we cluster our 2,280 customers into distinct segments using RFM (Recency, Frequency, Monetary value) metrics?
- What are the defining profiles of each customer segment, and what targeted actions (such as specific retention strategies for drifting, high-value customers) should we implement for them?
- What are the projected weekly sales when forecasted by product category or store location?
- Does our forecasting model successfully outperform a naive baseline (e.g., achieving a lower MAPE than a "same as last week" approach)?

In [9]:
import pandas as pd
from pathlib import Path

DATA = Path("..") /"Data"/ "kenya_retail"

tx = pd.read_csv(DATA / "transactions.csv", parse_dates=["date"])
stores = pd.read_csv(DATA / "stores.csv")
customers = pd.read_csv(DATA / "customers.csv", parse_dates=["signup_date"])
products = pd.read_csv(DATA / "products.csv")

for name, df in [("transactions", tx), ("stores", stores),
                 ("customers", customers), ("products", products)]:
    print(name, df.shape)

transactions (1802309, 16)
stores (60, 6)
customers (2280, 7)
products (62849, 5)


In [10]:
print("duplicate rows:", tx.duplicated().sum())
tx = tx.drop_duplicates()

print("missing payment method:", tx["payment_method"].isna().sum())
tx["payment_method"] = tx["payment_method"].fillna("Unknown")

print("missing customer ID:", tx["customer_id"].isna().mean().round(3))

duplicate rows: 3581
missing payment method: 5407
missing customer ID: 0.079


In [11]:
df = (tx
      .merge(stores.rename(columns={"county": "store_county"}), on="store_id", how="left")
      .merge(products, on="product_id", how="left")
      .merge(customers[["customer_id", "county", "loyalty_tier"]]
             .rename(columns={"county": "customer_county"}),
             on="customer_id", how="left"))
df.head()

,line_id,basket_id,date,hour,store_id,customer_id,product_id,quantity,unit_price_kes,gross_sales_kes,...,tier,format,size_sqm,opened_year,category,product_group,product_subgroup,brand,customer_county,loyalty_tier
0,1,26984851472,2024-01-01,16,KE-ST042,C02166,1004906,1,140.0,140.0,...,Rural,Mini-market,448,2013,PRODUCE,POTATOES,POTATOES RUSSET (BULK&BAG),Private,Bungoma,Silver
1,2,26984851472,2024-01-01,16,KE-ST042,C02166,1033142,1,60.0,60.0,...,Rural,Mini-market,448,2013,PRODUCE,ONIONS,ONIONS SWEET (BULK&BAG),National,Bungoma,Silver
2,3,26984851472,2024-01-01,16,KE-ST042,C02166,1036325,1,90.0,90.0,...,Rural,Mini-market,448,2013,PRODUCE,VEGETABLES - ALL OTHERS,CELERY,Private,Bungoma,Silver
3,4,26984851472,2024-01-01,16,KE-ST042,C02166,1082185,1,85.0,85.0,...,Rural,Mini-market,448,2013,PRODUCE,TROPICAL FRUIT,BANANAS,National,Bungoma,Silver
4,5,26984851472,2024-01-01,16,KE-ST042,C02166,8160430,1,135.0,135.0,...,Rural,Mini-market,448,2013,PRODUCE,ORGANICS FRUIT & VEGETABLES,ORGANIC CARROTS,Private,Bungoma,Silver


In [12]:
df.groupby("store_county")["net_sales_kes"].sum().sort_values(ascending=False).head(10)

store_county
Nairobi        58858289.0
Bungoma        51605051.0
Kakamega       37379286.0
Machakos       34665143.0
Kiambu         28740040.0
Trans Nzoia    20089244.0
Laikipia       18918355.0
Uasin Gishu    17145132.0
Kisumu         16619146.0
Murang'a       16517704.0
Name: net_sales_kes, dtype: float64

In [14]:
df.head()

,line_id,basket_id,date,hour,store_id,customer_id,product_id,quantity,unit_price_kes,gross_sales_kes,...,tier,format,size_sqm,opened_year,category,product_group,product_subgroup,brand,customer_county,loyalty_tier
0,1,26984851472,2024-01-01,16,KE-ST042,C02166,1004906,1,140.0,140.0,...,Rural,Mini-market,448,2013,PRODUCE,POTATOES,POTATOES RUSSET (BULK&BAG),Private,Bungoma,Silver
1,2,26984851472,2024-01-01,16,KE-ST042,C02166,1033142,1,60.0,60.0,...,Rural,Mini-market,448,2013,PRODUCE,ONIONS,ONIONS SWEET (BULK&BAG),National,Bungoma,Silver
2,3,26984851472,2024-01-01,16,KE-ST042,C02166,1036325,1,90.0,90.0,...,Rural,Mini-market,448,2013,PRODUCE,VEGETABLES - ALL OTHERS,CELERY,Private,Bungoma,Silver
3,4,26984851472,2024-01-01,16,KE-ST042,C02166,1082185,1,85.0,85.0,...,Rural,Mini-market,448,2013,PRODUCE,TROPICAL FRUIT,BANANAS,National,Bungoma,Silver
4,5,26984851472,2024-01-01,16,KE-ST042,C02166,8160430,1,135.0,135.0,...,Rural,Mini-market,448,2013,PRODUCE,ORGANICS FRUIT & VEGETABLES,ORGANIC CARROTS,Private,Bungoma,Silver


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1798728 entries, 0 to 1798727
Data columns (total 27 columns):
 #   Column            Dtype         
---  ------            -----         
 0   line_id           int64         
 1   basket_id         int64         
 2   date              datetime64[ns]
 3   hour              int64         
 4   store_id          object        
 5   customer_id       object        
 6   product_id        int64         
 7   quantity          int64         
 8   unit_price_kes    float64       
 9   gross_sales_kes   float64       
 10  discount_kes      float64       
 11  net_sales_kes     float64       
 12  promo_flag        int64         
 13  promo_type        object        
 14  payment_method    object        
 15  payment_ref       object        
 16  store_county      object        
 17  tier              object        
 18  format            object        
 19  size_sqm          int64         
 20  opened_year       int64         
 21  category